In [1]:
import stpsf
from astropy.io import fits
from tqdm import tqdm

In [2]:
path_image1 = "/home/wesley/Desktop/JWST_REU/2025_stage_2_data/"
fits_file = '/home/wesley/Desktop/JWST_REU/2025_stage_2_data/nan_rm_jwst_2025_f444w_f470n_d1_frame_A.fits'

In [ ]:
shift_amount = .01
test_size = 40 
coarse_shift_values = []

for p in range(-test_size, test_size + 1, +1):
    for j in range(test_size, -test_size - 1, -1):
        coarse_shift_values.append([j*shift_amount, p*shift_amount])

shift_amount = .0005
test_size = 80
fine_shift_values = []

for p in range(-test_size, test_size + 1, +1):
    for j in range(test_size, -test_size - 1, -1):
        fine_shift_values.append([j*shift_amount, p*shift_amount])

In [ ]:
coarse_y_shifts = [item[0] for item in coarse_shift_values]
coarse_x_shifts = [item[1] for item in coarse_shift_values]
fine_y_shifts = [item[0] for item in fine_shift_values]
fine_x_shifts = [item[1] for item in fine_shift_values]

In [ ]:
coarse_psf_stack = []
fine_psf_stack = []

In [6]:
inst = stpsf.setup_sim_to_match_file(fits_file, verbose=False)

In [ ]:
for k in tqdm(range(len(coarse_shift_values))):
    inst.options['source_offset_x'] = coarse_shift_values[k][1]
    inst.options['source_offset_y'] = coarse_shift_values[k][0]
    
    boxsize = 25

    sim_psf   = inst.calc_psf(fov_pixels=boxsize)
    sim_psf_data    = sim_psf["DET_DIST"].data
    coarse_psf_stack.append(sim_psf_data)

In [ ]:
hdus1 = [fits.PrimaryHDU()]

for k, psf in enumerate(coarse_psf_stack):
    hdu1 = fits.ImageHDU(data=psf)

    hdu1.header["XSHIFT"] = (float(coarse_x_shifts[k]), "X shift [arcseconds]")
    hdu1.header["YSHIFT"] = (float(coarse_y_shifts[k]), "Y shift [arcseconds]")

    hdus1.append(hdu1)
hdul1 = fits.HDUList(hdus1)
hdul1.writeto("coarse_psf_stack_b25.fits", overwrite=True)

In [9]:
for k in tqdm(range(len(fine_shift_values))):
    inst.options['source_offset_x'] = fine_shift_values[k][1]
    inst.options['source_offset_y'] = fine_shift_values[k][0]
    
    boxsize = 25

    sim_psf   = inst.calc_psf(fov_pixels=boxsize)
    sim_psf_data    = sim_psf["DET_DIST"].data
    fine_psf_stack.append(sim_psf_data)


100%|██████████| 25921/25921 [4:22:41<00:00,  1.64it/s]  


In [10]:
hdus2 = [fits.PrimaryHDU()]

for k, psf in enumerate(fine_psf_stack):
    hdu2 = fits.ImageHDU(data=psf)

    hdu2.header["XSHIFT"] = (float(fine_x_shifts[k]), "X shift [arcseconds]")
    hdu2.header["YSHIFT"] = (float(fine_y_shifts[k]), "Y shift [arcseconds]")

    hdus2.append(hdu2)
hdul2 = fits.HDUList(hdus2)
hdul2.writeto("fine_psf_stack_b25.fits", overwrite=True)